# 02 · WCA Domain Rules

WCA-Bench cannot be evaluated correctly unless the competition rules are decoded explicitly.
This notebook exercises the four rule families that the pipeline handles:

1. Sentinel values (`-1` DNF, `-2` DNS, `0` no result)
2. Value decoding by `format` (`time` = centiseconds, `number` = moves, `multi` = composite)
3. Multi-blind encodings (`1SSAATTTTT` and `0DDTTTTTMM`) with a round-trip invariant
4. Round-format normalization (`best of 3`, `average of 5` with trim-best-and-worst, `mean of 3`)

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from wca_bench.data import decoders as dec

print("DNF =", dec.DNF, "| DNS =", dec.DNS, "| NO_RESULT =", dec.NO_RESULT)

## 1. Sentinel values

Sentinels never participate in averaging; they are counted separately so that DNF rates can be
modelled as an imbalanced classification problem (Task 3).

In [ ]:
for value in (-1, -2, 0, 8653, 28):
    label = {dec.DNF: "DNF", dec.DNS: "DNS", dec.NO_RESULT: "NO RESULT"}.get(value, "score")
    formatted = dec.format_time_centiseconds(value) if value > 0 else "-"
    print(f"{value:>7} -> {label:<10} formatted={formatted}")

## 2. Value decoding by format

`8653` in `time` format is 1:26.53. In `number` format the same digits would be 8653 moves, which
is why decoding must always be driven by the event's `format`.

In [ ]:
print("time  :", dec.decode_result_value(8653, "time"))
print("number:", dec.decode_result_value(8653, "number"))
print("centiseconds 8653 ==", dec.format_time_centiseconds(8653))

## 3. Multi-blind encoding

Multi-blind results pack solved / attempted counts and elapsed seconds into one integer.
The round-trip invariant `encode(decode(v)) == v` is asserted in `tests/unit/test_decoders.py`.

In [ ]:
import random

random.seed(0)
ok = True
for _ in range(200):
    solved = random.randint(0, 15)
    attempted = random.randint(solved, solved + 6)
    seconds = random.randint(60, 3600)
    for version in ("old", "new"):
        encoded = dec.encode_multi(solved, attempted, seconds, version=version)
        got = dec.decode_multi(encoded)
        if (got.solved, got.attempted, got.seconds) != (solved, attempted, seconds):
            ok = False
            print("MISMATCH", version, solved, attempted, seconds, encoded, got)

print("round-trip invariant holds for 200 random cases:", ok)
print("example (old):", dec.encode_multi(5, 7, 1200, version="old"), "->", dec.decode_multi(dec.encode_multi(5, 7, 1200, version="old")))

## 4. Round-format normalization

For `average of 5` the best and worst attempts are removed before averaging. A single DNF therefore
often disappears; two DNFs collapse the whole round. That asymmetry is a first-class feature for
Task 2 and Task 3.

In [ ]:
no_dnf = [1000, 1100, 1200, 1300, 1400]
one_dnf = [1000, 1100, 1200, 1300, dec.DNF]
two_dnf = [1000, 1100, 1200, dec.DNF, dec.DNF]

for name, attempts in (("ao5 clean", no_dnf), ("ao5 1 DNF", one_dnf), ("ao5 2 DNF", two_dnf)):
    try:
        avg = dec.compute_average(attempts, "a")
    except Exception as exc:  # noqa: BLE001
        avg = f"<{type(exc).__name__}: {exc}>"
    print(f"{name:<12} attempts={attempts} -> average={avg}")

print("\nbest of 3  :", dec.compute_best([1200, 1000, 1100]))
print("mean of 3  :", dec.compute_average([1200, 1000, 1100], "m"))

## 5. Scramble normalization

In the TSV export, the newlines separating the individual 3x3 scrambles of a `333mbf` attempt are
replaced by `|`. The pipeline restores them.

In [ ]:
raw = "R U R' U'|F2 L2 D|B' D2 F'"
normalized = dec.normalize_multiblind_scramble(raw)
print(repr(normalized))
print("scramble count:", len([line for line in normalized.splitlines() if line.strip()]))